In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications import EfficientNetV2L
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

strategy = tf.distribute.MirroredStrategy()
print("Number of devices:", strategy.num_replicas_in_sync)

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


I0000 00:00:1785343271.008453      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785343271.011048      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
DATA_DIR = "/kaggle/input/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/Lung Disease Dataset"

train_dir = os.path.join(DATA_DIR, "train")
val_dir   = os.path.join(DATA_DIR, "val")
test_dir  = os.path.join(DATA_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32       
NUM_CLASSES = 5

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_generator.class_indices)

Found 6054 images belonging to 5 classes.
Found 2016 images belonging to 5 classes.
Found 2025 images belonging to 5 classes.
Class indices: {'Bacterial Pneumonia': 0, 'Corona Virus Disease': 1, 'Normal': 2, 'Tuberculosis': 3, 'Viral Pneumonia': 4}


In [3]:
with strategy.scope():
    base_model = EfficientNetV2L(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(512, activation='relu'),
        Dropout(0.4),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

model.summary()

473176280/473176280 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-l (Functional)   │ (None, 7, 7, 1280)     │   117,746,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,540,453 (452.20 MB)

 Trainable params: 791,045 (3.02 MB)

 Non-trainable params: 117,749,408 (449.18 MB)

In [4]:
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1)
checkpoint = ModelCheckpoint("best_efficientnetv2l.keras", monitor='val_accuracy', save_best_only=True, verbose=1)

history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

E0000 00:00:1785343406.511760      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1/efficientnetv2-l_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 801ms/step - accuracy: 0.2391 - loss: 1.7486INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Epoch 1: val_accuracy improved from None to 0.41121, saving model to best_efficientnetv2l.keras

Epoch 1: finished saving model to best_efficientnetv2l.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 261s 1s/step - accuracy: 0.2554 - loss: 1.6919 - val_accuracy: 0.4112 - val_loss: 1.5836 - learning_rate: 1.0000e-04
Epoch 2/15
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 694ms/step - accuracy: 0.2750 - loss: 1.6186
Epoch 2: val_accuracy improved from 0.41121 to 0.48611, saving model to best_efficientnetv2l.keras

Epoch 2: finished saving model to best_efficientnetv2l.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 162s 847ms/step - accuracy: 0.2887 - loss: 1.5949 - val_accur

In [7]:
with strategy.scope():
    base_model.trainable = True
    
    for layer in base_model.layers[:-80]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

print("Model recompiled for fine-tuning")

Model recompiled for fine-tuning


In [8]:
history_ft = model.fit(
    train_generator,
    epochs=12,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/12
INFO:tensorflow:Collective all_reduce tensors: 76 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1785345991.671558      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1/efficientnetv2-l_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 720ms/step - accuracy: 0.2915 - loss: 1.6415
Epoch 1: val_accuracy did not improve from 0.54514
190/190 ━━━━━━━━━━━━━━━━━━━━ 234s 959ms/step - accuracy: 0.3295 - loss: 1.5512 - val_accuracy: 0.5253 - val_loss: 1.1929 - learning_rate: 1.0000e-05
Epoch 2/12
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 720ms/step - accuracy: 0.3841 - loss: 1.4289
Epoch 2: val_accuracy did not improve from 0.54514
190/190 ━━━━━━━━━━━━━━━━━━━━ 161s 844ms/step - accuracy: 0.3819 - loss: 1.4267 - val_accuracy: 0.5159 - val_loss: 1.1534 - learning_rate: 1.0000e-05
Epoch 3/12
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 719ms/step - accuracy: 0.4057 - loss: 1.3752
Epoch 3: val_accuracy improved from 0.54514 to 0.57788, saving model to best_efficientnetv2l.keras

Epoch 3: finished saving model to best_efficientnetv2l.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 167s 876ms/step - accuracy: 0.4153 - loss: 1.3565 - val_accuracy: 0.5779 - val_loss: 1.0840 - learning_rate: 1.0000e-05
Epoch 4/12
190/190 ━━━━━━━━━━━━━━━━━━

In [9]:
with strategy.scope():
    base_model.trainable = True
    
    for layer in base_model.layers[:-200]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=5e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

print("Unfroze last 200 layers")

Unfroze last 200 layers


In [ ]:
history_ft2 = model.fit(
    train_generator,
    epochs=12,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/12
INFO:tensorflow:Collective all_reduce tensors: 76 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1785347698.610477      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1/efficientnetv2-l_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 729ms/step - accuracy: 0.4122 - loss: 1.3934
Epoch 1: val_accuracy did not improve from 0.57788
190/190 ━━━━━━━━━━━━━━━━━━━━ 237s 965ms/step - accuracy: 0.4106 - loss: 1.3777 - val_accuracy: 0.5268 - val_loss: 1.1794 - learning_rate: 5.0000e-05
Epoch 2/12
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 721ms/step - accuracy: 0.4415 - loss: 1.3125
Epoch 2: ReduceLROnPlateau reducing learning rate to 1.4999999621068127e-05.

Epoch 2: val_accuracy did not improve from 0.57788
190/190 ━━━━━━━━━━━━━━━━━━━━ 161s 846ms/step - accuracy: 0.4511 - loss: 1.2856 - val_accuracy: 0.4648 - val_loss: 1.2397 - learning_rate: 5.0000e-05
Epoch 3/12
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 720ms/step - accuracy: 0.4655 - loss: 1.2422
Epoch 3: val_accuracy improved from 0.57788 to 0.58978, saving model to best_efficientnetv2l.keras

Epoch 3: finished saving model to best_efficientnetv2l.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 169s 886ms/step - accuracy: 0.4798 - loss: 1.2181 - val_accuracy: 0.5898 - val_